# ObjectiveとPosteriorTransform：多出力GPから意思決定へ

2出力GPから1つの評価指標を作る際、**事後分布を変換する `ScalarizedPosteriorTransform`** と、**事後サンプルを評価する `LinearMCObjective` / `GenericMCObjective`** の違いを学びます。

BoTorchの既存公開APIを使い、線形・非線形の候補提案、観測、GP再学習、次の候補提案まで実行します。Notebook内で独自の獲得関数クラスは定義しません。


## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.analytic import LogExpectedImprovement
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import (
    GenericMCObjective,
    LinearMCObjective,
    ScalarizedPosteriorTransform,
)
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### このNotebookで区別する3つのもの

1. **モデル出力**：GPが予測する2つの応答 `y1, y2`。
2. **評価指標（utility）**：応答を使って「良い条件」を1つの数値で評価するルール。
3. **獲得関数**：GPの予測と不確実性を使って「次にどこを測るか」を決めるルール。

`utility` を定義することと `acquisition` を新規実装することは別です。ここでは既存の `LogExpectedImprovement` / `qLogExpectedImprovement` を利用します。

## 2. Two-output surrogate

2出力を予測する`SingleTaskGP`を作ります。ここではmulti-objective Pareto optimizationではなく、2出力から**scalar decision utility**を作る問題です。

In [ ]:
def outcomes(X: torch.Tensor) -> torch.Tensor:
    first = torch.sin(2 * torch.pi * X)
    second = torch.cos(2 * torch.pi * X) + 0.2 * X
    return torch.cat([first, second], dim=-1)


train_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
train_Y = outcomes(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

probe_X = torch.tensor([[0.25], [0.75]])
with torch.no_grad():
    raw_posterior = model.posterior(probe_X)
print("raw posterior mean shape:", raw_posterior.mean.shape)

## 3. PosteriorTransform acts before sampling

線形scalarizationなら`ScalarizedPosteriorTransform`でmulti-output posteriorそのものをscalar posteriorへ変換できます。meanだけでなくcovarianceも線形写像後のposteriorとして扱われます。

### 線形スカラー化の意味

重み `weights=[0.7,0.3]` は、2つの応答を `0.7*y1 + 0.3*y2` として評価する設定です。**目的の単位やスケールが違う場合は、重みを設定する前に意味を整理**する必要があります。

`ScalarizedPosteriorTransform` は線形変換後の平均と共分散を持つ**スカラーの事後分布**を作ります。GPの出力次元が2でも、解析的な単目的獲得関数に渡せるようになります。

In [ ]:
weights = torch.tensor([0.7, 0.3])
posterior_transform = ScalarizedPosteriorTransform(weights=weights)

with torch.no_grad():
    transformed = model.posterior(
        probe_X,
        posterior_transform=posterior_transform,
    )
    expected = posterior_transform(raw_posterior)

torch.testing.assert_close(transformed.mean, expected.mean)
torch.testing.assert_close(transformed.variance, expected.variance)
print("transformed posterior mean shape:", transformed.mean.shape)

## 4. Analytic acquisition after PosteriorTransform

posteriorがscalarになったため、compatibleなanalytic acquisitionへ接続できます。

### `best_f` は何を表すか

`LogExpectedImprovement` の `best_f` は、改善の基準となる**観測済みの最良スカラー評価値**です。この例では `train_Y @ weights` を計算してから最大値を取ります。

`best_f` はGPの事後平均の最大値ではありません。観測ノイズが大きい場合は、ノイズを考慮する獲得関数の選択も検討します。

In [ ]:
scalar_train_Y = train_Y @ weights
analytic_acq = LogExpectedImprovement(
    model=model,
    best_f=scalar_train_Y.max(),
    posterior_transform=posterior_transform,
)
analytic_candidate, analytic_value = optimize_acqf(
    acq_function=analytic_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("PosteriorTransform candidate:", analytic_candidate)
print("LogEI value:", analytic_value)

## 5. MC Objective acts on posterior samples

`LinearMCObjective`はposterior distributionを変換するのではなく、sampling後の`[..., q, m]` samplesをscalar utility `[..., q]`へ変換します。

In [ ]:
sampler = SobolQMCNormalSampler(torch.Size([128]), seed=123)
linear_objective = LinearMCObjective(weights=weights)

with torch.no_grad():
    samples = sampler(model.posterior(probe_X))
    objective_samples = linear_objective(samples)
print("posterior samples shape:", samples.shape)
print("objective samples shape:", objective_samples.shape)

### MC Objectiveはサンプルを評価する

`LinearMCObjective` は、事後分布から得た各サンプル `[..., q, 2]` をスカラー値 `[..., q]` に変換します。**分布そのものを別のGaussian分布に変換する操作ではありません**。

線形効用なら解析的PosteriorTransformとMC Objectiveは同じ評価ルールを表せますが、MC法にはサンプリング誤差があるため、候補位置が数値的に完全一致するとは限りません。

## 6. MC acquisition with the linear objective

In [ ]:
mc_acq = qLogExpectedImprovement(
    model=model,
    best_f=scalar_train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=321),
    objective=linear_objective,
)
mc_candidate, mc_value = optimize_acqf(
    acq_function=mc_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("LinearMCObjective candidate:", mc_candidate)
print("qLogEI value:", mc_value)

## 7. Nonlinear utility belongs on the MC-objective path

非線形utilityは一般にGaussian posteriorをGaussianのまま保つとは限りません。ここでは`GenericMCObjective`でsamples上のutilityを定義します。

In [ ]:
def nonlinear_utility(Y: torch.Tensor, X: torch.Tensor | None = None) -> torch.Tensor:
    del X
    return Y[..., 0] - 0.25 * Y[..., 1].square()


nonlinear_objective = GenericMCObjective(nonlinear_utility)
# Improvement threshold is defined by utilities of actually observed outcomes.
nonlinear_best_f = nonlinear_utility(train_Y).max()

nonlinear_acq = qLogExpectedImprovement(
    model=model,
    best_f=nonlinear_best_f,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=654),
    objective=nonlinear_objective,
)
nonlinear_candidate, nonlinear_value = optimize_acqf(
    acq_function=nonlinear_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("nonlinear-objective candidate:", nonlinear_candidate)
print("qLogEI value:", nonlinear_value)

### 非線形効用と改善基準

ここでは `y1 - 0.25*y2²` を評価指標にします。`y2` が大きくても、2乗のペナルティで評価が下がることがあります。非線形変換後の分布は一般にGaussianとは限らないため、**サンプルを変換する `GenericMCObjective`** を使います。

改善基準は観測済みの `train_Y` に非線形効用を適用して計算します。モデルからサンプリングした予測値を観測値の代わりに使いません。

## 8. Candidate evaluation in raw outcome space

optimizerが返すのは入力`X`です。実験・simulationではraw outcomesを観測し、その後decision utilityを計算します。

In [ ]:
candidate_X = nonlinear_candidate
candidate_Y = outcomes(candidate_X)
candidate_utility = nonlinear_utility(candidate_Y)
updated_X = torch.cat([train_X, candidate_X], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate X:", candidate_X)
print("raw candidate Y:", candidate_Y)
print("decision utility:", candidate_utility)
print("updated shapes:", updated_X.shape, updated_Y.shape)

### 実験では2つの生の応答を測定する

最適化器が返すのは入力 `X` です。実験で得られるのは `y1` と `y2` の2つの生の測定値であり、評価指標はその後に計算します。

効用だけを学習する単出力モデルに置き換えることも考えられますが、このNotebookでは**多出力GPを維持し、意思決定側で効用を指定する構成**を学びます。

## 9. Visualize the different scalar utilities

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
grid_Y = outcomes(grid)
linear_utility = grid_Y @ weights
nonlinear_values = nonlinear_utility(grid_Y)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), linear_utility, label="linear utility")
ax.plot(grid.squeeze(-1), nonlinear_values, label="nonlinear utility")
ax.scatter(
    analytic_candidate.item(),
    (outcomes(analytic_candidate) @ weights).item(),
    marker="x",
    s=80,
    label="PosteriorTransform candidate",
)
ax.scatter(
    nonlinear_candidate.item(),
    nonlinear_utility(outcomes(nonlinear_candidate)).item(),
    marker="x",
    s=80,
    label="nonlinear MC candidate",
)
ax.legend()
plt.show()

### 追加観測後にモデルを再学習する

新しい候補を評価したら、**2つの生の応答をそのまま訓練データに追加**します。再学習後に同じ `GenericMCObjective` と新しい観測最良値を使って獲得関数を作り直し、次の候補を提案します。

In [ ]:
updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()
updated_best_f = nonlinear_utility(updated_Y).max()
next_acq = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_best_f,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=655),
    objective=nonlinear_objective,
)
next_candidate, next_value = optimize_acqf(
    acq_function=next_acq, bounds=bounds, q=1,
    num_restarts=4, raw_samples=64,
)
print("更新後の観測最良効用:", updated_best_f.item())
print("更新後の次の候補:", next_candidate.item())
print("更新後のqLogEI:", next_value.item())
assert updated_X.shape[0] == train_X.shape[0] + 1
assert next_candidate.shape == (1, 1)

### 2出力の予測はどのように変わるか

候補を1点追加して再学習した前後の予測平均を比較します。目的値は最終的に1つの効用へまとめますが、**モデル自体は2出力の予測を保持**しています。

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    before_mean = model.posterior(grid).mean
    after_mean = updated_model.posterior(grid).mean
true_grid_Y = outcomes(grid)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for k, ax in enumerate(axes):
    ax.plot(grid.squeeze(-1), true_grid_Y[:, k], label="True outcome")
    ax.plot(grid.squeeze(-1), before_mean[:, k], label="Before fit")
    ax.plot(grid.squeeze(-1), after_mean[:, k], label="After fit")
    ax.scatter(candidate_X.squeeze(-1), candidate_Y[:, k], marker="x", s=75, label="New observation")
    ax.set(xlabel="x", title=f"Output {k+1}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 10. Responsibility map

| Concern | Correct layer |
| --- | --- |
| affine transform of posterior distribution | `PosteriorTransform` |
| scalar utility of posterior samples | `MCAcquisitionObjective` |
| nonlinear sample utility | `GenericMCObjective` |
| Pareto / hypervolume vector objective | multi-objective acquisition path |
| unknown outcome feasibility | acquisition outcome constraints |
| known feasibility of X | acquisition optimizer constraints |
| candidate input perturbations | uncertainty / scenario generation |
| risk across environmental scenarios | risk aggregation after objective evaluation |

## 11. Important boundaries

- `PosteriorTransform`とMC `Objective`をAPI統一のために同一抽象化しません。
- native BoTorch objectsをrobotorchan-specific wrapperへ変換しません。
- nonlinear sample utilityを`PosteriorTransform`として偽装しません。
- qLogEHVI/qLogNEHVIではPareto vectorを保持するため、scalarizationがdefaultではありません。
- `ScalarizedPosteriorTransform`対応はmodel posterior contractに依存します。ALEBO等には明示的restrictionがあります。
- output reduction中のspecialized modelではoriginal output space復元との順序が重要です。
- robust input perturbationとrisk aggregationはPhase 21のrobust workflowで扱います。

## 12. Related documentation

- `docs/optimization/objective-posterior-transform.md`
- `docs/theory/acquisition/01_foundations.md`
- `docs/theory/07_multitask_multioutput.md`
- `33_multiobjective_acquisition.ipynb`

Objectiveは「posterior outputsを何のutilityとして使うか」、PosteriorTransformは「posterior distributionをどう変換するか」という異なる責務を持ちます。

## 13. 既存APIの使い分け

| 目的 | BoTorch / robotorchan API | 役割 |
|---|---|---|
| 2出力GPの学習 | `SingleTaskGP.fit()` | 各応答の事後分布を学習 |
| 線形な分布変換 | `ScalarizedPosteriorTransform` | 解析的獲得関数向けのスカラー事後分布 |
| 線形なサンプル評価 | `LinearMCObjective` | MCサンプルを重み付き和へ変換 |
| 非線形なサンプル評価 | `GenericMCObjective` | MCサンプルから効用を計算 |
| 単一点の解析的改善 | `LogExpectedImprovement` | スカラー事後分布に対する改善 |
| MCによる改善 | `qLogExpectedImprovement` | サンプル上の効用に対する改善 |
| 候補最適化 | `optimize_acqf` | 次の入力条件を選択 |

**重要：** `PosteriorTransform` と `MC Objective` は同じ抽象化ではありません。前者は事後分布の変換、後者はサンプルに対する評価です。非線形効用を無理に解析的Gaussian posteriorとして扱わないようにします。